In [ ]:
# 按下 Ctrl + Shift + P (Mac: Cmd + Shift + P)。

# 输入 Developer: Reload Window 并回车。

# 作用： 这会强制刷新 VS Code 的 UI 连接，通常重连后能恢复控制，或者至少让你能重新启动内核。

import os 
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import make_grid
from tqdm import tqdm
from datetime import datetime
import argparse

device = torch.device('cuda:5' if torch.cuda.is_available() else 'cpu')
local_rank = 0
print("device:",device)

import utils
seed=42

import random
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
# from models import Clipper
# clip_extractor = Clipper("ViT-L/14", hidden_state=False, norm_embs=True, device=device)
imsize = 512

device: cuda:5


In [ ]:
import os
from PIL import Image
import torch
import numpy as np


source_dir = '/EEG_Image_decode/Generation/generated_imgs/NotebookGCNsub-08FM_test10_epoch10_num_inference_steps20Cos0'
target_dir = '/EEG_Image_decode/Generation/generated_imgs/NotebookGCNsub-08FM_test10_epoch10_num_inference_steps20Cos0_tensor'

# Create the target directory if it doesn't exist
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

# Initialize a list to hold all the image tensors
tensor_list = []

# Iterate over the folders in the source directory
for folder_name in sorted(os.listdir(source_dir)):
    folder_path = os.path.join(source_dir, folder_name)
    
    # Check if it's a directory
    if os.path.isdir(folder_path):
        # Iterate over the images in the folder
        for image_name in sorted(os.listdir(folder_path)):
            image_path = os.path.join(folder_path, image_name)
            
            # Load the image
            with Image.open(image_path) as img:
                # Convert the image to a PyTorch tensor and add a batch dimension
                tensor = torch.tensor(np.array(img)).unsqueeze(0)
                tensor_list.append(tensor)

# Concatenate all tensors along the 0th dimension
all_tensors = torch.cat(tensor_list, dim=0)

# Save the combined tensor
combined_tensor_path = os.path.join(target_dir, "all_images.pt")
torch.save(all_tensors, combined_tensor_path)

In [ ]:
import os
from PIL import Image
import torch
import numpy as np



source_dir = 'EEG2Image/THINGS-Data/THINGS-EEG_images_set/test_images'
target_dir = 'EEG2Image/THINGS-Data/THINGS-EEG_images_set/test_images_tensor'



# Create the target directory if it doesn't exist
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

# Initialize a list to hold all the image tensors
tensor_list = []

# Iterate over the folders in the source directory
for folder_name in sorted(os.listdir(source_dir)):
    folder_path = os.path.join(source_dir, folder_name)
    
    # Check if it's a directory
    if os.path.isdir(folder_path):
        # Iterate over the images in the folder
        for image_name in sorted(os.listdir(folder_path)):
            image_path = os.path.join(folder_path, image_name)
            
            # Load the image
            with Image.open(image_path) as img:
                # Convert the image to a PyTorch tensor and add a batch dimension
                tensor = torch.tensor(np.array(img)).unsqueeze(0)
                tensor_list.append(tensor)
                print(tensor)

# Concatenate all tensors along the 0th dimension
all_tensors = torch.cat(tensor_list, dim=0)

# Save the combined tensor
combined_tensor_path = os.path.join(target_dir, "all_images.pt")
torch.save(all_tensors, combined_tensor_path)

# Configurations

In [ ]:

recon_path = '/EEG2Image/EEG_Image_decode/Generation/generated_imgs/NotebookGCNsub-08FM_test10_epoch10_num_inference_steps20Cos0_tensor/all_images.pt'

all_images_path = '/EEG2Image/THINGS-Data/THINGS-EEG_images_set/test_images_tensor/all_images.pt'



import torch
from torchvision import transforms


all_brain_recons = torch.load(f'{recon_path}', map_location='cpu')
all_images = torch.load(f'{all_images_path}', map_location='cpu')

print("Original shapes:", all_images.shape, all_brain_recons.shape)



if all_images.ndim == 4 and all_images.shape[-1] == 3:
    all_images = all_images.permute(0, 3, 1, 2)
    
if all_brain_recons.ndim == 4 and all_brain_recons.shape[-1] == 3:
    all_brain_recons = all_brain_recons.permute(0, 3, 1, 2)

imsize = 256
resizer = transforms.Resize((imsize, imsize))

all_images = resizer(all_images)
all_brain_recons = resizer(all_brain_recons)

print("Processed shapes:", all_images.shape, all_brain_recons.shape)
print('recon_path',recon_path)


In [ ]:
def compare_images(original, reconstructed, titles=["Original", "Reconstructed"]):

    # 处理图像张量
    images = []
    for img in [original, reconstructed]:
        if img.is_cuda:
            img = img.cpu()
        if img.dtype == torch.uint8:
            img = img.float() / 255.0
        elif img.max() > 1.0:
            img = img / 255.0
        img = torch.clamp(img, 0, 1)
        # 转换维度顺序
        if img.dim() == 3 and img.shape[0] in [1, 3]:
            img = img.permute(1, 2, 0)
        images.append(img)
    
    # 创建对比图
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(img.numpy())
        axes[i].set_title(title)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# 使用示例（假设有 all_images 变量）
compare_images(all_images[4], all_brain_recons[40])

# Display reconstructions next to ground truth images

In [ ]:
imsize = 256
# 将所有图像和脑部重建图像调整为指定大小
all_images = transforms.Resize((imsize,imsize))(all_images)
all_brain_recons = transforms.Resize((imsize,imsize))(all_brain_recons)

print( 'all_images.shape,all_brain_recons.shape\n', all_images.shape,all_brain_recons.shape )
# 设置随机种子以确保结果可重现
np.random.seed(0)
# 定义要显示的图像索引序列，并进行翻转 
ind = np.flip(np.array([1,11,21,55,175,86, 96, 111,196,122 ,2 ,10 ,28, 44 ,99]))

# 创建交错排列的图像张量，用于后续可视化展示
all_interleaved = torch.zeros(len(ind)*2,3,imsize,imsize)
icount = 0
# 按照指定索引顺序，交替放置原始图像和对应的脑部重建图像
for t in ind:
    if torch.max(all_images[t]).item() > 1:
        all_interleaved[icount] = all_images[t].float() / 255.0
        all_interleaved[icount+1] = all_brain_recons[t*10].float() / 255.0
    else:        
        all_interleaved[icount] = all_images[t]
        all_interleaved[icount+1] = all_brain_recons[t*10]
    icount += 2

# 设置matplotlib保存图像时的边界参数
plt.rcParams["savefig.bbox"] = 'tight'

def show(imgs, figsize):

    if not isinstance(imgs, list):
        imgs = [imgs]

    fig, axs = plt.subplots(ncols=len(imgs), squeeze=False, figsize=figsize)

    for i, img in enumerate(imgs):
        img = img.detach()

        img = transforms.ToPILImage()(img)

        axs[0, i].imshow(np.asarray(img))

        axs[0, i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

grid = make_grid(all_interleaved, nrow=10, padding=2)

show(grid, figsize=(20,16))




# FID


In [ ]:
# ...existing code...
import torch
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from scipy import linalg
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def compute_fid_from_tensors(all_images, all_brain_recons, device=None, batch_size=32, eps=1e-6):

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    recon_sub = all_brain_recons[::10]
    assert recon_sub.shape[0] == all_images.shape[0], "长度不匹配：检查1:10映射"

    weights = Inception_V3_Weights.DEFAULT

    inc = create_feature_extractor(inception_v3(weights=weights, aux_logits=True), return_nodes=['avgpool']).to(device)
    inc.eval()
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)
    def extract_feats(x):
        feats = []
        for i in range(0, len(x), batch_size):
            b = x[i:i+batch_size].to(device).float()
            # ensure in [0,1]
            if b.max() > 1.5:
                b = b / 255.0
            # resize to 299
            b = F.interpolate(b, size=(299,299), mode='bilinear', align_corners=False)
            # normalize
            b = (b - mean) / std
            out = inc(b)['avgpool']  # shape [B,2048,1,1]
            out = out.reshape(out.shape[0], -1).cpu().numpy()
            feats.append(out)
        return np.concatenate(feats, axis=0)

    real_feats = extract_feats(all_images)
    fake_feats = extract_feats(recon_sub)

    # compute mean / cov
    mu1 = np.mean(real_feats, axis=0)
    mu2 = np.mean(fake_feats, axis=0)
    sigma1 = np.cov(real_feats, rowvar=False)
    sigma2 = np.cov(fake_feats, rowvar=False)

    # frechet distance
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        # numerical fix
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
    # if imaginary due to numerical error, take real part
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)
    return float(fid)

# 使用示例（在 notebook 环境已存在 all_images, all_brain_recons）
fid_value = compute_fid_from_tensors(all_images, all_brain_recons, device=device, batch_size=16)
print(f"FID: {fid_value:.4f}")


#NotebookGCNsub-08FM_test10_epoch10_ep10_clip826  FID: 167.8485


# 2-Way Identification

In [ ]:
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names

@torch.no_grad()
def two_way_identification(all_brain_recons, all_images, model, preprocess, feature_layer=None, return_avg=True):


    preds = model(torch.stack([preprocess(recon) for recon in all_brain_recons], dim=0).to(device))
    reals = model(torch.stack([preprocess(indiv) for indiv in all_images], dim=0).to(device))

    if feature_layer is None:
        preds = preds.float().flatten(1).cpu().numpy()
        reals = reals.float().flatten(1).cpu().numpy()
    else:
        preds = preds[feature_layer].float().flatten(1).cpu().numpy()
        reals = reals[feature_layer].float().flatten(1).cpu().numpy()


    r = np.corrcoef(reals, preds)
    r = r[:len(all_images), len(all_images):]
    congruents = np.diag(r)


    success = r < congruents
    success_cnt = np.sum(success, 0)


    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images)-1)
        return perf
    else:
        return success_cnt, len(all_images)-1
    
    
    

## PixCorr

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
])


all_images_flattened = preprocess(all_images).reshape(len(all_images), -1).cpu()
all_brain_recons_flattened = preprocess(all_brain_recons).reshape(len(all_brain_recons), -1).cpu()

print(all_images_flattened.shape)
print(all_brain_recons_flattened.shape)

corrsum = 0

data_length = all_images_flattened.shape[0]
for i in tqdm(range(data_length)):
    corrsum += np.corrcoef(all_images_flattened[i], all_brain_recons_flattened[i*10])[0][1]
corrmean = corrsum / data_length

pixcorr = corrmean
print(pixcorr)



## SSIM

In [ ]:

from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim

preprocess = transforms.Compose([
    transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR), 
])


img_gray = rgb2gray(preprocess(all_images).permute((0,2,3,1)).cpu())
recon_gray = rgb2gray(preprocess(all_brain_recons).permute((0,2,3,1)).cpu())
print("converted, now calculating ssim...")


recon_gray_sub = recon_gray[::10]
assert img_gray.shape[0] == recon_gray_sub.shape[0], "数量不一致，无法一一对应"

print('img_gray.shape, recon_gray_sub.shape\n',img_gray.shape, recon_gray_sub.shape)
ssim_score=[]
for im,rec in tqdm(zip(img_gray,recon_gray_sub),total=len(all_images)):
    ssim_score.append(ssim(rec, im, multichannel=True, gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=1.0))

ssim = np.mean(ssim_score)
print(ssim)



In [13]:
all_brain_recons = all_brain_recons.float() / 255.0
all_images = all_images.float() / 255.0
N = len(all_images)
all_brain_recons_sub = all_brain_recons[::10]  # 只取每10个的第一个

### InceptionV3

In [ ]:
def extract_features_in_batches(images, model, preprocess, device, feature_layer=None, batch_size=32):
    feats = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
            batch = torch.stack([preprocess(img) for img in batch], dim=0).to(device)
            out = model(batch)
            # print(out)
            if feature_layer is not None:
                out = out[feature_layer]
            feats.append(out.float().flatten(1).cpu())
    return torch.cat(feats, dim=0)

@torch.no_grad()
def two_way_identification(all_brain_recons, all_images, model, preprocess, feature_layer=None, return_avg=True, batch_size=32):
    preds = extract_features_in_batches(all_brain_recons, model, preprocess, device, feature_layer, batch_size)
    reals = extract_features_in_batches(all_images, model, preprocess, device, feature_layer, batch_size)
    preds = preds.numpy()
    reals = reals.numpy()
    r = np.corrcoef(reals, preds)
    r = r[:len(all_images), len(all_images):]
    congruents = np.diag(r)
    success = r < congruents
    success_cnt = np.sum(success, 0)
    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images)-1)
        return perf
    else:
        return success_cnt, len(all_images)-1    
    
from torchvision.models import inception_v3, Inception_V3_Weights
weights = Inception_V3_Weights.DEFAULT
inception_model = create_feature_extractor(inception_v3(weights=weights), 
                                           return_nodes=['avgpool']).to(device)
inception_model.eval().requires_grad_(False)

# see weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
print('all_images shape' , all_images.shape,'  \n  ' ,all_images[0,0,0,:5])
print('all_brain_recons shape' , all_brain_recons_sub.shape,'  \n  ' ,all_brain_recons_sub[0,0,0,:5] )


all_per_correct = two_way_identification(
    all_brain_recons_sub, all_images, inception_model, preprocess, 'avgpool', batch_size=32
)

inception = np.mean(all_per_correct)
print(f"2-way Percent Correct: {inception:.4f}")


### CLIP

In [ ]:
# import clip
# clip_model, preprocess = clip.load("ViT-L/14", device=device)
def extract_features_in_batches(images, model, preprocess, device, feature_layer=None, batch_size=32):
    feats = []
    if hasattr(model, "eval"):
        model.eval()
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
            batch = torch.stack([preprocess(img) for img in batch], dim=0).to(device)
            out = model(batch)
            # 兼容 open_clip 返回 tuple 的情况
            if isinstance(out, tuple):
                out = out[0]
            if feature_layer is not None and isinstance(out, dict):
                out = out[feature_layer]
            feats.append(out.float().flatten(1).cpu())
    return torch.cat(feats, dim=0)

@torch.no_grad()
def two_way_identification(all_brain_recons, all_images, model, preprocess, feature_layer=None, return_avg=True, batch_size=32):
    preds = extract_features_in_batches(all_brain_recons, model, preprocess, device, feature_layer, batch_size)
    reals = extract_features_in_batches(all_images, model, preprocess, device, feature_layer, batch_size)
    preds = preds.numpy()
    reals = reals.numpy()
    r = np.corrcoef(reals, preds)
    r = r[:len(all_images), len(all_images):]
    congruents = np.diag(r)
    success = r < congruents
    success_cnt = np.sum(success, 0)
    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images)-1)
        return perf
    else:
        return success_cnt, len(all_images)-1  
import os
# 使用 hf-mirror 镜像 下载clip权重
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

model_type = 'ViT-H-14'
import open_clip

clip_model, preprocess_train, feature_extractor = open_clip.create_model_and_transforms(
    model_type, pretrained='laion2b_s32b_b79k', precision='fp32', device = device)

preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

print('all_images shape' , all_images.shape,'  \n  ' ,all_images[0,0,0,:5])
print('all_brain_recons shape' , all_brain_recons_sub.shape,'  \n  ' ,all_brain_recons_sub[0,0,0,:5] )

all_per_correct = two_way_identification(all_brain_recons_sub, all_images,
                                        clip_model,preprocess, None) # final layer
clip_ = np.mean(all_per_correct)
print(f"2-way Percent Correct: {clip_:.4f}") 


### Efficient Net

In [ ]:
import os
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
weights = EfficientNet_B1_Weights.DEFAULT
eff_model = create_feature_extractor(efficientnet_b1(weights=weights), 
                                    return_nodes=['avgpool']).to(device)
eff_model.eval().requires_grad_(False)

# see weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
print('all_images shape' , all_images.shape,'  \n  ' ,all_images[0,0,0,:5])
print('all_brain_recons shape' , all_brain_recons_sub.shape,'  \n  ' ,all_brain_recons_sub[0,0,0,:5] )

gt = eff_model(preprocess(all_images).to(device))['avgpool']
gt = gt.reshape(len(gt),-1).cpu().numpy()
fake = eff_model(preprocess(all_brain_recons_sub).to(device))['avgpool']
fake = fake.reshape(len(fake),-1).cpu().numpy()

effnet = np.array([sp.spatial.distance.correlation(gt[i],fake[i]) for i in range(len(gt))]).mean()
print("Distance:",effnet)


### SwAV

In [ ]:
swav_model = torch.hub.load('facebookresearch/swav:main', 'resnet50')
swav_model = create_feature_extractor(swav_model, 
                                    return_nodes=['avgpool']).to(device)
swav_model.eval().requires_grad_(False)

preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('all_images shape' , all_images.shape,'  \n  ' ,all_images[0,0,0,:5])
print('all_brain_recons shape' , all_brain_recons_sub.shape,'  \n  ' ,all_brain_recons_sub[0,0,0,:5] )

gt = swav_model(preprocess(all_images).to(device))['avgpool']
gt = gt.reshape(len(gt),-1).cpu().numpy()
fake = swav_model(preprocess(all_brain_recons_sub).to(device))['avgpool']
fake = fake.reshape(len(fake),-1).cpu().numpy()

swav = np.array([sp.spatial.distance.correlation(gt[i],fake[i]) for i in range(len(gt))]).mean()
print("Distance:",swav)
